<a href="https://colab.research.google.com/github/dcdlima/Miscelaneous/blob/main/Busca_Artigos_Cient%C3%ADficos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
"""
Script para busca de artigos científicos com operadores lógicos.
Utiliza APIs: CrossRef, arXiv e Semantic Scholar
"""

import requests
import json
import csv
from datetime import datetime
from typing import List, Dict, Tuple
from urllib.parse import quote
import time

!pip install pyeuropepmc

class BuscadorArtigosCientificos:
    """Classe para buscar artigos científicos em bases de dados públicas"""

    def __init__(self):
        self.artigos_encontrados = []
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }

    def buscar_crossref(self, query: str, limit: int = 50) -> List[Dict]:
        """
        Busca artigos no CrossRef

        Args:
            query: String de busca
            limit: Número máximo de resultados

        Returns:
            Lista de artigos encontrados
        """
        artigos = []
        url = "https://api.crossref.org/works"

        try:
            print(f"\n🔍 Buscando no CrossRef: {query[:60]}...")

            params = {
                'query': query,
                'rows': limit,
                'sort': 'relevance',
                'order': 'desc'
            }

            response = requests.get(url, params=params, timeout=10, headers=self.headers)
            response.raise_for_status()

            data = response.json()

            if 'message' in data and 'items' in data['message']:
                for item in data['message']['items']:
                    artigo = {
                        'titulo': item.get('title', [''])[0] if isinstance(item.get('title'), list) else item.get('title', 'N/A'),
                        'autores': ', '.join([f"{a.get('given', '')} {a.get('family', '')}"
                                            for a in item.get('author', [])[:3]]) or 'N/A',
                        'ano': item.get('published-online', {}).get('date-parts', [[None]])[0][0] or item.get('issued', {}).get('date-parts', [[None]])[0][0],
                        'doi': item.get('DOI', 'N/A'),
                        'resumo': item.get('abstract', 'N/A'),
                        'fonte': 'CrossRef',
                        'url': f"https://doi.org/{item.get('DOI')}" if item.get('DOI') else 'N/A',
                        'relevancia': 0.8
                    }
                    artigos.append(artigo)

            print(f"✓ {len(artigos)} artigos encontrados no CrossRef")

        except requests.exceptions.RequestException as e:
            print(f"✗ Erro ao buscar no CrossRef: {e}")
        except json.JSONDecodeError as e:
            print(f"✗ Erro ao processar resposta do CrossRef: {e}")

        return artigos

    def buscar_arxiv(self, query: str, limit: int = 50) -> List[Dict]:
        """
        Busca artigos no arXiv

        Args:
            query: String de busca
            limit: Número máximo de resultados

        Returns:
            Lista de artigos encontrados
        """
        artigos = []
        url = "http://export.arxiv.org/api/query"

        try:
            print(f"\n🔍 Buscando no arXiv: {query[:60]}...")

            # Formatar query para arXiv
            query_arxiv = f"all:{query}"

            params = {
                'search_query': query_arxiv,
                'max_results': limit,
                'sortBy': 'relevance',
                'sortOrder': 'descending'
            }

            response = requests.get(url, params=params, timeout=10, headers=self.headers)
            response.raise_for_status()

            # Parse XML response
            import xml.etree.ElementTree as ET
            root = ET.fromstring(response.content)

            # Namespace do arXiv
            ns = {'arxiv': 'http://arxiv.org/schemas/atom',
                  'atom': 'http://www.w3.org/2005/Atom'}

            entries = root.findall('atom:entry', ns)

            for entry in entries:
                titulo = entry.find('atom:title', ns)
                autores = entry.findall('atom:author', ns)
                publicado = entry.find('atom:published', ns)
                resumo = entry.find('atom:summary', ns)
                arxiv_id = entry.find('atom:id', ns)

                artigo = {
                    'titulo': titulo.text.strip() if titulo is not None else 'N/A',
                    'autores': ', '.join([a.find('atom:name', ns).text
                                        for a in autores[:3]]) if autores else 'N/A',
                    'ano': int(publicado.text[:4]) if publicado is not None else 'N/A',
                    'doi': 'arXiv',
                    'resumo': resumo.text.strip() if resumo is not None else 'N/A',
                    'fonte': 'arXiv',
                    'url': arxiv_id.text if arxiv_id is not None else 'N/A',
                    'relevancia': 0.75
                }
                artigos.append(artigo)

            print(f"✓ {len(artigos)} artigos encontrados no arXiv")

        except requests.exceptions.RequestException as e:
            print(f"✗ Erro ao buscar no arXiv: {e}")
        except Exception as e:
            print(f"✗ Erro ao processar resposta do arXiv: {e}")

        return artigos

    def buscar_europepmc(self, query: str, limit: int = 50) -> List[Dict]:
        """
        Busca artigos no Europe PMC

        Args:
            query: String de busca
            limit: Número máximo de resultados

        Returns:
            Lista de artigos encontrados
        """
        artigos = []
        url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search?query={query}&format=json"

        try:
            print(f"\n🔍 Buscando no Europe PMC: {query[:60]}...")

            params = {
                'query': query,
                'pageSize': limit,
                'format': 'json',
                'sortBy': 'RELEVANCE'
            }

            response = requests.get(url, params=params, timeout=10, headers=self.headers)
            response.raise_for_status()

            data = response.json()

            if 'resultList' in data and 'result' in data['resultList']:
                for item in data['resultList']['result']:
                    artigo = {
                        'titulo': item.get('title', 'N/A'),
                        'autores': ', '.join([f"{a.get('firstName', '')} {a.get('lastName', '')}"
                                            for a in item.get('authorList', {}).get('author', [])[:3]]) or 'N/A',
                        'ano': item.get('pubYear', 'N/A'),
                        'doi': item.get('doi', 'N/A'),
                        'resumo': item.get('abstractText', 'N/A'),
                        'fonte': 'Europe PMC',
                        'url': f"https://europepmc.org/article/{item.get('id')}" if item.get('id') else 'N/A',
                        'relevancia': 0.85
                    }
                    artigos.append(artigo)

            print(f"✓ {len(artigos)} artigos encontrados no Europe PMC")

        except requests.exceptions.RequestException as e:
            print(f"✗ Erro ao buscar no Europe PMC: {e}")
        except json.JSONDecodeError as e:
            print(f"✗ Erro ao processar resposta do Europe PMC: {e}")

        return artigos

    def combinar_e_classificar(self, artigos: List[Dict], top_n: int = 20) -> List[Dict]:
        """
        Remove duplicatas e classifica artigos por relevância

        Args:
            artigos: Lista de artigos encontrados
            top_n: Número de top artigos a retornar

        Returns:
            Lista classificada dos top artigos
        """
        # Remover duplicatas por título
        artigos_unicos = {}
        for artigo in artigos:
            titulo_normalizado = artigo['titulo'].lower().strip()
            if titulo_normalizado not in artigos_unicos:
                artigos_unicos[titulo_normalizado] = artigo

        # Ordenar por relevância
        artigos_ordenados = sorted(
            artigos_unicos.values(),
            key=lambda x: x['relevancia'],
            reverse=True
        )

        return artigos_ordenados[:top_n]

    def executar_busca(self, grupos_busca: Tuple[List[str], List[str], List[str]],
                       top_n: int = 20) -> List[Dict]:
        """
        Executa a busca combinando os três grupos com operadores lógicos

        Args:
            grupos_busca: Tupla com 3 listas de termos (OR dentro, AND entre)
            top_n: Número de artigos para retornar

        Returns:
            Lista dos top artigos encontrados
        """
        grupo1, grupo2, grupo3 = grupos_busca

        # Construir queries com operadores lógicos
        query1 = ' OR '.join([f'"{termo}"' for termo in grupo1])
        query2 = ' OR '.join([f'"{termo}"' for termo in grupo2])
        query3 = ' OR '.join([f'"{termo}"' for termo in grupo3])

        # Query combinada (AND entre grupos)
        query_completa = f'({query1}) AND ({query2}) AND ({query3})'

        print(f"\n{'='*80}")
        print(f"QUERY COMPLETA:\n{query_completa}")
        print(f"{'='*80}")

        # Buscar em todas as fontes
        todos_artigos = []

        todos_artigos.extend(self.buscar_arxiv(query_completa))
        time.sleep(1)  # Respeitar rate limit

        todos_artigos.extend(self.buscar_crossref(query_completa))
        time.sleep(1)

        todos_artigos.extend(self.buscar_europepmc(query_completa))

        # Combinar e classificar
        self.artigos_encontrados = self.combinar_e_classificar(todos_artigos, top_n)

        return self.artigos_encontrados

    def gerar_relatorio_texto(self, filepath: str = 'relatorio_artigos.txt') -> None:
        """Gera relatório em formato texto"""

        with open(filepath, 'w', encoding='utf-8') as f:
            f.write("="*100 + "\n")
            f.write("RELATÓRIO DE BUSCA DE ARTIGOS CIENTÍFICOS\n")
            f.write("="*100 + "\n\n")

            f.write(f"Data: {datetime.now().strftime('%d/%m/%Y às %H:%M:%S')}\n")
            f.write(f"Total de artigos encontrados: {len(self.artigos_encontrados)}\n\n")

            f.write("-"*100 + "\n")

            for idx, artigo in enumerate(self.artigos_encontrados, 1):
                f.write(f"\n{idx}. {artigo['titulo']}\n")
                f.write("-" * 100 + "\n")
                f.write(f"   Autores: {artigo['autores']}\n")
                f.write(f"   Ano: {artigo['ano']}\n")
                f.write(f"   Fonte: {artigo['fonte']}\n")
                f.write(f"   DOI/ID: {artigo['doi']}\n")
                f.write(f"   URL: {artigo['url']}\n")
                f.write(f"   Relevância: {artigo['relevancia']:.2%}\n")
                if artigo['resumo'] != 'N/A':
                    f.write(f"   Resumo: {artigo['resumo'][:300]}...\n")
                f.write("\n")

            f.write("="*100 + "\n")

        print(f"\n✓ Relatório em texto salvo em: {filepath}")

    def gerar_relatorio_csv(self, filepath: str = 'relatorio_artigos.csv') -> None:
        """Gera relatório em formato CSV"""

        if not self.artigos_encontrados:
            print("Nenhum artigo para gerar relatório")
            return

        with open(filepath, 'w', newline='', encoding='utf-8') as f:
            campos = ['Posição', 'Título', 'Autores', 'Ano', 'Fonte', 'DOI/ID',
                      'URL', 'Relevância', 'Resumo']
            writer = csv.DictWriter(f, fieldnames=campos)

            writer.writeheader()

            for idx, artigo in enumerate(self.artigos_encontrados, 1):
                writer.writerow({
                    'Posição': idx,
                    'Título': artigo['titulo'],
                    'Autores': artigo['autores'],
                    'Ano': artigo['ano'],
                    'Fonte': artigo['fonte'],
                    'DOI/ID': artigo['doi'],
                    'URL': artigo['url'],
                    'Relevância': f"{artigo['relevancia']:.2%}",
                    'Resumo': artigo['resumo'][:200] if artigo['resumo'] != 'N/A' else 'N/A'
                })

        print(f"✓ Relatório em CSV salvo em: {filepath}")

    def gerar_relatorio_json(self, filepath: str = 'relatorio_artigos.json') -> None:
        """Gera relatório em formato JSON"""

        dados = {
            'data_busca': datetime.now().isoformat(),
            'total_artigos': len(self.artigos_encontrados),
            'artigos': self.artigos_encontrados
        }

        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(dados, f, ensure_ascii=False, indent=2)

        print(f"✓ Relatório em JSON salvo em: {filepath}")

    def exibir_resumo(self) -> None:
        """Exibe um resumo dos resultados no console"""

        print("\n" + "="*80)
        print("RESUMO DOS RESULTADOS")
        print("="*80)
        print(f"\nTotal de artigos encontrados: {len(self.artigos_encontrados)}\n")

        for idx, artigo in enumerate(self.artigos_encontrados, 1):
            print(f"{idx:2d}. [{artigo['fonte']}] {artigo['titulo'][:70]}")
            print(f"    Autores: {artigo['autores'][:60]}")
            print(f"    Ano: {artigo['ano']} | Relevância: {artigo['relevancia']:.0%}\n")


def main():
    """Função principal"""

    # Definir os grupos de busca com operadores OR
    grupos = (
        # Grupo 1: Prospecção/Foresight Tecnológico
        [
            "Technological Foresight",
            "Technology Prospecting",
            "Prospecção Tecnológica",
            "Tech Foresight"
        ],
        # Grupo 2: Modelos de linguagem e IA generativa
        [
            "Retrieval-Augmented Generation",
            "RAG",
            "Large Language Model",
            "LLM",
            "Generative AI",
            "GPT",
            "BERT",
            "Transformer"
        ],
        # Grupo 3: Literatura científica e descoberta de conhecimento
        [
            "Scientific Literature",
            "Systematic Review",
            "Bibliometric",
            "Knowledge Discovery",
            "Text Mining",
            "NLP",
            "Information Retrieval"
        ]
    )

    # Criar buscador
    buscador = BuscadorArtigosCientificos()

    # Executar busca
    print("\n🚀 Iniciando busca de artigos científicos...\n")
    artigos = buscador.executar_busca(grupos, top_n=20)

    # Exibir resumo
    buscador.exibir_resumo()

    # Gerar relatórios em múltiplos formatos
    print("\n📄 Gerando relatórios...\n")
    buscador.gerar_relatorio_texto('relatorio_artigos.txt')
    buscador.gerar_relatorio_csv('relatorio_artigos.csv')
    buscador.gerar_relatorio_json('relatorio_artigos.json')

    print("\n✓ Busca concluída com sucesso!\n")


if __name__ == "__main__":
    main()



🚀 Iniciando busca de artigos científicos...


QUERY COMPLETA:
("Technological Foresight" OR "Technology Prospecting" OR "Prospecção Tecnológica" OR "Tech Foresight") AND ("Retrieval-Augmented Generation" OR "RAG" OR "Large Language Model" OR "LLM" OR "Generative AI" OR "GPT" OR "BERT" OR "Transformer") AND ("Scientific Literature" OR "Systematic Review" OR "Bibliometric" OR "Knowledge Discovery" OR "Text Mining" OR "NLP" OR "Information Retrieval")

🔍 Buscando no arXiv: ("Technological Foresight" OR "Technology Prospecting" OR "P...
✓ 0 artigos encontrados no arXiv

🔍 Buscando no CrossRef: ("Technological Foresight" OR "Technology Prospecting" OR "P...
✓ 50 artigos encontrados no CrossRef

🔍 Buscando no Europe PMC: ("Technological Foresight" OR "Technology Prospecting" OR "P...
✓ 50 artigos encontrados no Europe PMC

RESUMO DOS RESULTADOS

Total de artigos encontrados: 20

 1. [Europe PMC] Distribution-to-Points Matching for Image Text Retrieval.
    Autores: N/A
    Ano: 2026 | Relev